In [ ]:
#libraries
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import RFE
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
import numpy as np
from duckdb import query as qr



# from src.data_fetchers import fetch_loop_habits_data, get_sheet_data
# from src.data_cleaners import clean_data

from src.data_fetchers import fetch_loop_habits_data, get_sheet_data,fetch_daily_steps
from src.data_cleaners import get_combined_data,prepare_sleep_data

habit_sleep_data = get_combined_data()

raw_sleep = get_sheet_data()
sleep_data = prepare_sleep_data(raw_sleep)


In [ ]:
#generate metadata for the modeling_set
habit_sleep_data.info()

# Create a data dictionary for the modeling_set table
data_dictionary = {
    'habit_date': 'Date of the habit entry',
    'meditate': 'Binary indicator if meditation was done (1) or not (0)',
    'exercise': 'Binary indicator if exercise was done (1) or not (0)',
    'morning': 'Score for morning habits',
    'day': 'Score for day habits',
    'evening': 'Score for evening habits',
    'drinks': 'Number of drinks consumed',
    'candy': 'Amount of candy consumed',
    'pen': 'Amount of pen units consumed',
    'video_games': 'Time spent playing video games',
    'day_of_week': 'Day of the week (0=Monday, 6=Sunday)',
    'day_label': 'Name of the day of the week',
    'week_number': 'Week number of the year',
    'month': 'Month of the year',
    'weekend': 'Binary indicator if the day is a weekend (1) or not (0)',
    'drinks_flag': 'Binary indicator if drinks were consumed (1) or not (0)',
    'candy_flag': 'Binary indicator if candy was consumed (1) or not (0)',
    'pen_flag': 'Binary indicator if pen was used (1) or not (0)',
    'video_games_flag': 'Binary indicator if video games were played (1) or not (0)',
    'total_score': 'Total score for the day',
    'morning_chart': 'Interpolated score for morning habits',
    'day_chart': 'Interpolated score for day habits',
    'evening_chart': 'Interpolated score for evening habits',
    'total_score_lag1': 'Total score of the previous day',
    'morning_lag1': 'Morning score of the previous day',
    'day_lag1': 'Day score of the previous day',
    'evening_lag1': 'Evening score of the previous day',
    'meditate_lag1': 'Meditation indicator of the previous day',
    'exercise_lag1': 'Exercise indicator of the previous day',
    'drinks_lag1': 'Drinks consumed on the previous day',
    'candy_lag1': 'Candy consumed on the previous day',
    'pen_lag1': 'Pen consumed  the previous day',
    'drinks_flag_lag1': 'Drinks flag of the previous day',
    'candy_flag_lag1': 'Candy flag of the previous day',
    'pen_flag_lag1': 'Pen flag of the previous day',
    'video_games_lag1': 'Video games time of the previous day',
    'total_score_chart': 'Sum of interpolated morning, day, and evening scores',
    'total_score_chart_r3': '3-day rolling average of total_score_chart',
    'sleep_score': 'Score of sleep quality for the night prior',
    'sleep_start_time': 'Time when sleep started',
    'time_to_fall_asleep': 'Time taken to fall asleep',
    'restful_time': 'Time spent in restful sleep',
    'restless_time': 'Time spent in restless sleep',
    'total_sleep_time': 'Total time spent sleeping. Restful + Restless',
    'bed_exit_time': 'Amount of time spent out of bed during the night', 
    'laydown_time': 'Time when lay down to sleep',
    'out_of_bed_time': 'Time when woke up / got out of bed',
    'out_of_bed_time_mins': 'Time when woke up based on minutes past midnight'
}

# Display the data dictionary
# for key, value in data_dictionary.items():
#     print(f'{key}: {value}')

# Include the dataframe name in the data dictionary display
print("Data Dictionary for modeling_set:")
for key, value in data_dictionary.items():
    print(f'{key}: {value}')

In [ ]:
import xgboost as xgb
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.inspection import PartialDependenceDisplay
import seaborn as sns
import math

# Prepare modeling dataset
modeling_set = habit_sleep_data.dropna(subset=['total_score'])
modeling_set = modeling_set.drop(columns=['habit_date', 'day_chart', 'morning_chart', 'evening_chart'
                                        ,'total_score_chart', 'total_score_chart_r3','month','day_label'
                                        ,'morning', 'day', 'evening', 'week_number'
                                        ,'drinks_flag','candy_flag', 'pen_flag','video_games_flag'
                                        ,'drinks_flag_lag1', 'candy_flag_lag1','pen_flag_lag1'
                                        #,'drinks_lag1', 'candy_lag1', 'pen_lag1'
                                        ,'sleep_start_time', 'laydown_time', 'out_of_bed_time'
                                        ,'drinks', 'candy', 'pen', 'meditate', 'exercise','video_games'
                                        ,'total_score_lag1'
                                        # ,'morning_lag1', 'day_lag1', 'evening_lag1'
                                        ])

# Select features and target variable
X = modeling_set.drop(columns=['total_score'])
y = modeling_set['total_score']

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)
xgb_model.fit(X_train, y_train)

# Get feature importance
gain = xgb_model.get_booster().get_score(importance_type='gain')
gain_df = pd.DataFrame({
    'Feature': gain.keys(),
    'Importance (gain)': gain.values()
}).sort_values('Importance (gain)', ascending=False)

# Create figures for different diagnostic plots
plt.style.use('seaborn')

# 1. Feature Importance Plot
fig1, ax1 = plt.subplots(figsize=(10, 6))
gain_df.plot(x='Feature', y='Importance (gain)', kind='bar', ax=ax1)
ax1.set_title('Feature Importance (Gain)')
ax1.set_xlabel('Features')
ax1.set_ylabel('Importance Score')
ax1.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

# 2. Partial Dependence Plots for top features
n_top_features = 12
top_features = gain_df['Feature'].head(n_top_features).tolist()

# Create subplot grid
fig2, axes = plt.subplots(math.ceil(n_top_features / 3), 3, figsize=(15, 12))

# Create all PDPs in one display
display = PartialDependenceDisplay.from_estimator(
    xgb_model,
    X_train,
    top_features,
    kind="average",
    grid_resolution=50,
    ax=axes.ravel()[:len(top_features)]
)

# Add histograms and adjust titles
for idx, feature in enumerate(top_features):
    ax = axes.ravel()[idx]
    
    # Add histogram on twin axis showing proportions
    ax2 = ax.twinx()
    counts, bins, _ = ax2.hist(X_train[feature], alpha=0.3, color='r', 
                              weights=np.ones_like(X_train[feature]) / len(X_train[feature]))
    
    # Ensure ylabel is on the right side
    ax2.yaxis.set_label_position("right")
    ax2.set_ylabel('Proportion of Observations', color='r', rotation=270, labelpad=15)
    ax2.tick_params(axis='y', colors='r')
    ax2.set_ylim(0, 1)  # Set y-axis from 0 to 1
    
    # Update title
    ax.set_title(f'Partial Dependence Plot for {feature}')

# Hide any empty subplots
for idx in range(len(top_features), len(axes.ravel())):
    axes.ravel()[idx].set_visible(False)

plt.tight_layout()
plt.show()

# 3. Residual Analysis
# y_pred_train = xgb_model.predict(X_train)
# residuals = y_train - y_pred_train

# fig3, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# # Residuals vs Predicted
# ax1.scatter(y_pred_train, residuals, alpha=0.5)
# ax1.axhline(y=0, color='r', linestyle='--')
# ax1.set_xlabel('Predicted Values')
# ax1.set_ylabel('Residuals')
# ax1.set_title('Residuals vs Predicted Values')

# # Residual Distribution
# sns.histplot(residuals, kde=True, ax=ax2)
# ax2.set_title('Distribution of Residuals')
# ax2.set_xlabel('Residual Value')

# plt.tight_layout()
# plt.show()

# 4. Feature Interaction Plot (for top 2 features)
if len(top_features) >= 2:
    feature1, feature2 = top_features[:2]
    
    # Create interaction grid
    f1_min, f1_max = X_train[feature1].min(), X_train[feature1].max()
    f2_min, f2_max = X_train[feature2].min(), X_train[feature2].max()
    
    f1_grid = np.linspace(f1_min, f1_max, 20)
    f2_grid = np.linspace(f2_min, f2_max, 20)
    xx, yy = np.meshgrid(f1_grid, f2_grid)
    
    # Create prediction grid
    X_grid = X_train.copy()
    grid_predictions = np.zeros((len(f1_grid), len(f2_grid)))
    
    for i, f1_val in enumerate(f1_grid):
        for j, f2_val in enumerate(f2_grid):
            X_grid[feature1] = f1_val
            X_grid[feature2] = f2_val
            grid_predictions[j, i] = xgb_model.predict(X_grid).mean()
    
    fig4, ax = plt.subplots(figsize=(10, 8))
    contour = ax.contourf(xx, yy, grid_predictions, levels=20, cmap='viridis')
    plt.colorbar(contour, ax=ax, label='Predicted Value')
    ax.set_xlabel(feature1)
    ax.set_ylabel(feature2)
    ax.set_title(f'Feature Interaction: {feature1} vs {feature2}')
    
    # Add scatter plot of actual data points
    ax.scatter(X_train[feature1], X_train[feature2], 
              c='white', alpha=0.5, s=10, label='Training Data')
    ax.legend()
    
    plt.tight_layout()
    plt.show()

print("\nFeature Importance Rankings:")
print(gain_df)

# do a partial dependence plot for restful_time and total_sleep_time
# Create subplot grid
fig2, axes = plt.subplots(1, 2, figsize=(15, 6))

# Create all PDPs in one display
display = PartialDependenceDisplay.from_estimator(
    xgb_model,
    X_train,
    ['restful_time', 'total_sleep_time'],
    kind="average",
    grid_resolution=50,
    ax=axes.ravel()
)

# Add histograms and adjust titles
for idx, feature in enumerate(['restful_time', 'total_sleep_time']):
    ax = axes.ravel()[idx]
    
    # Add histogram on twin axis showing proportions
    ax2 = ax.twinx()
    counts, bins, _ = ax2.hist(X_train[feature], alpha=0.3, color='r', 
                              weights=np.ones_like(X_train[feature]) / len(X_train[feature]))
    
    # Ensure ylabel is on the right side
    ax2.yaxis.set_label_position("right")
    ax2.set_ylabel('Proportion of Observations', color='r', rotation=270, labelpad=15)
    ax2.tick_params(axis='y', colors='r')
    ax2.set_ylim(0, 1)  # Set y-axis from 0 to 1
    
    # Update title
    ax.set_title(f'Partial Dependence Plot for {feature}')

plt.tight_layout()
plt.show()



In [ ]:
from sklearn.tree import DecisionTreeRegressor, plot_tree


# Weekend vs Weekday comparison
print("\nWeekend vs Weekday Scores:")
print(modeling_set.groupby('weekend')['total_score'].agg(['mean', 'std', 'count']))

# Correlation matrix
correlation_matrix = modeling_set.corr()['total_score'].sort_values(ascending=False)
print("\nCorrelations with Total Score:")
print(correlation_matrix)

# Correlation heatmap
# plt.figure(figsize=(12, 8))
# sns.heatmap(modeling_set.corr(), annot=True, cmap='coolwarm', center=0)
# plt.title('Correlation Heatmap')
# plt.tight_layout()
# plt.show()

# Analyze how previous day's activities affect today's score
lag_cols = [col for col in modeling_set.columns if 'lag1' in col]
lag_correlations = modeling_set[['total_score'] + lag_cols].corr()['total_score'].sort_values(ascending=False)
print("\nCorrelations with Previous Day's Activities:")
print(lag_correlations)

# Find optimal thresholds for numeric variables
# numeric_cols = ['video_games', 'drinks_lag1', 'candy_lag1', 'pen_lag1']

# for col in numeric_cols:
#     # Create bins and analyze average score in each bin
#     modeling_set['bins'] = pd.qcut(modeling_set[col], q=5)
#     avg_by_bin = modeling_set.groupby('bins')['total_score'].mean()
#     print(f"\nAverage scores by {col} quintiles:")
#     print(avg_by_bin)
#     modeling_set.drop('bins', axis=1, inplace=True)

# Train a shallow decision tree for interpretable rules
# dt = DecisionTreeRegressor(max_depth=3, min_samples_leaf=5)
# dt.fit(X_train, y_train)

# # Visualize the decision tree
# plt.figure(figsize=(15, 10))
# plot_tree(dt, feature_names=X.columns, filled=True, rounded=True)
# plt.title('Decision Tree Rules for Daily Scores')
# plt.show()


In [ ]:
import seaborn as sns# steps_data
# habit_sleep_data

#I want descriptive statistics for the steps variable in habit_sleep_data
print("\nDescriptive Statistics for Steps:")
print(habit_sleep_data['steps'].describe())

#show a histogram of the steps variable
plt.figure(figsize=(10, 6))
sns.histplot(habit_sleep_data['steps'], bins=20, kde=True)
plt.title('Distribution of Steps')
plt.xlabel('Number of Steps')
plt.ylabel('Frequency')
plt.show()

In [12]:
from src.common import xicor

xicor(habit_sleep_data['total_sleep_time'], habit_sleep_data['restless_time'])

(nan, nan)